# Map Prodcom product codes to the corresponding HS 2022 codes

## Notebook README

This work is licenced under the Creative Commons Attribution (CC-BY 4.0) public licence.

**Related publication**

If you utilize any portions of the code, results, or draw inspiration for your projects, please reference the published article below.

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Code author: Nicolas LIENART
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This noebook is an helper to automatically match a series of PRODCOM codes to their corresponding 6-digit Harmonized System (HS) 2022 codes. Most of this notebook's code was generated using Anthropic Claude Sonnet 5. You can access the original prompts by following the link in the Relevant references section. Prodcom data does not specify wich version of the Prodcom claissification is used in the underlying data, it was inferred that it was probably Prodcom 2024 as of the time of this work, hence we selected the `CN2024_PRODCOM2024` correspondence file for better results. Resulting HS codes need to be checked against BACI's codes to make sure they are all present, otherwise user is invited to adjust non-matched codes manually.

**Updates**

 - August 14, 2026: Added the file.
 - August 15, 2026: Improve output by showing classification labels for PRODCOM and HS.

**Package versions**

 - See [`environment.yml`](./environment.yml)

**Relevant references**

- Gaulier, G., & Zignago, S. (2010). BACI: International trade database at the product-level. The 1994-2007 version (Working Papers Nos. 2010–23). CEPII. http://www.cepii.fr/CEPII/fr/publications/wp/abstract.asp?NoDoc=2726
- https://showvoc.op.europa.eu/#/datasets/ESTAT_Combined_Nomenclature__2024__CN_2024/unknown/data
- https://claude.ai/share/94cc184a-03fc-4de8-b04a-6319891ae1a3
- https://taxation-customs.ec.europa.eu/customs/common-customs-tariff-cct/tariff-classification-goods/combined-nomenclature_en
- https://showvoc.op.europa.eu/#/datasets/ESTAT_PRODCOM_List_2024/unknown/data

## Claude explanations about CN-PRODCOM correspondence

NOTE: Contrary to what is stated by Claude, we use CN2024-PRODCOM2024 correspondence!

> Prodcom <-> HS2022 (6-digit) lookup, built from the CN2026-PRODCOM2026
> xkos correspondence table published on ShowVoc / data.europa.eu.
> 
> How the mapping works
> 
> The correspondence file is a set of xkos:ConceptAssociation resources, each
> linking one CN2026 concept (xkos:sourceConcept) to one PRODCOM2026 concept
> (xkos:targetConcept), e.g.:
> 
> ```
> CN2026_PRODCOM2026_260111000080
>     xkos:sourceConcept  cn2026/260111000080   (CN8 "26011100" + "0080" suffix)
>     xkos:targetConcept  prodcom2026/07100010
> ```
> 
> The CN concept URI's local name is the 8-digit CN (Combined Nomenclature)
> code followed by a fixed 4-digit statistical suffix ("0080"). The first 6
> digits of the CN8 code ARE, by construction, the HS subheading: the
> Combined Nomenclature is HS-compliant and simply adds 2 extra digits to the
> 6-digit HS code. CN2026 is still built on the HS2022 edition (the next HS
> revision, HS2027, only enters into force on 1 January 2027 -- see WCO,
> "HS 2027 amendments", https://www.wcoomd.org/en/topics/nomenclature/instrument-and-tools/hs-nomenclature-2022-edition/hs-nomenclature-2022-edition.aspx),
> so "first 6 digits of the CN8 code in a CN2026 table" = "HS2022 subheading".
> This is an inference from the data/nomenclature timeline, not a field
> present in the file -- verify against an official CN2026 <-> HS2022
> concordance if this matters for your use case.
> 
> The mapping is many-to-one from CN8 to Prodcom (each CN8 code has exactly
> one Prodcom code in this file), but many-to-many once you aggregate to
> HS6: a single Prodcom code can legitimately correspond to several HS6
> subheadings, and a single HS6 subheading can correspond to several Prodcom
> codes. So the lookup functions below return sets, not single codes.
> 
> Requires: rdflib
> 
> Source: 14-08-2026 - https://claude.ai/share/94cc184a-03fc-4de8-b04a-6319891ae1a3


## Python library imports

In [65]:
from collections import defaultdict
import rdflib
import pandas as pd

## HS classification

To obtain the CSV file `Prodcom_data/HS classification based on CN2024.csv` we had to extract data from the EU ShowVoc platform. We used Claude as we had no previous knowledge of SPARQL query language.

We used this dataset: https://showvoc.op.europa.eu/#/datasets/ESTAT_Combined_Nomenclature__2024__CN_2024/unknown/sparql

SPARQL query to run in the "Sparql" tab:
```sql
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX adms: <http://www.w3.org/ns/adms#>
PREFIX : <http://data.europa.eu/xsp/cn2024/>

SELECT ?Member ?HS_Code ?Label_EN ?Explanatory_EN WHERE {
    :hs_subheadings skos:member ?Member .
    ?Member skos:prefLabel ?Label_EN .
    OPTIONAL { ?Member adms:identifier ?HS_Code }
    OPTIONAL { ?Member skos:scopeNote ?Explanatory_EN . FILTER (LANG(?Explanatory_EN) = "en") }
    FILTER (LANG(?Label_EN) = "en")
}
ORDER BY ?HS_Code
```

Claude discussion: https://claude.ai/share/cc0666ec-49b2-4228-a623-4f4f0ed6d4d2

### Import HS (CN) classification data

In [66]:
cn2024 = pd.read_csv("Prodcom_data/HS classification based on CN2024.csv", dtype={'HS_Code':str})
cn2024["HS_Code"] = cn2024["HS_Code"].transform(lambda x: x.replace(".", ""))
# cn2024

## PRODCOM classification

We took inspiration from the HS SPARQL code to derive a similar query to obtain the PRODCOM classification.

We used this dataset: https://showvoc.op.europa.eu/#/datasets/ESTAT_PRODCOM_List_2024/unknown/sparql

SPARQL query to run in the "Sparql" tab:
```sql
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX : <http://data.europa.eu/qw1/prodcom2024/>

SELECT ?Member ?Code ?Label_EN ?Explanatory_EN WHERE {
    :headings_prodcom skos:member ?Member .
    ?Member skos:prefLabel ?Label_EN .
    OPTIONAL { ?Member skos:notation ?Code }
    OPTIONAL { ?Member skos:altLabel ?Explanatory_EN . FILTER (LANG(?Explanatory_EN) = "en") }
    FILTER (LANG(?Label_EN) = "en")
} 
ORDER BY ?Code
```

### Import PRODCOM classification data

In [67]:
prodcom2024 = pd.read_csv("Prodcom_data/PRODCOM2024 classification.csv", dtype={'Code':str})
prodcom2024["Code"] = prodcom2024["Code"].transform(lambda x: x.replace(".", ""))
# prodcom2024

## CN-PRODCOM correspondence code

In [68]:
XKOS_SOURCE = rdflib.URIRef("http://rdf-vocabulary.ddialliance.org/xkos#sourceConcept")
XKOS_TARGET = rdflib.URIRef("http://rdf-vocabulary.ddialliance.org/xkos#targetConcept")
XKOS_ASSOC = rdflib.URIRef("http://rdf-vocabulary.ddialliance.org/xkos#ConceptAssociation")

In [69]:
def _local_name(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1]


def _normalize_prodcom(code: str) -> str:
    """Strip dots/spaces so '10.11.11.40' and '10111140' match the same key."""
    return code.replace(".", "").replace(" ", "").strip().upper()


In [70]:
class ProdcomHsCorrespondence:
    def __init__(self, path: str, rdf_format: str = None):
        """
        path: path to the correspondence file (any RDF serialization
              rdflib supports: json-ld, xml, turtle, nt, nquads, trig, n3...)
        rdf_format: rdflib format string. If None, rdflib guesses from the
              file extension (works for .jsonld, .ttl, .nt, .nq, .trig, .xml).
        """
        g = rdflib.Graph()
        g.parse(path, format=rdf_format)

        self.prodcom_to_hs6 = defaultdict(set)
        self.hs6_to_prodcom = defaultdict(set)
        self.prodcom_to_cn8 = defaultdict(set)  # keep full CN8 too, in case you need it

        # every ConceptAssociation has exactly one source and one target concept
        for assoc in g.subjects(rdflib.RDF.type, XKOS_ASSOC):
            src = next(g.objects(assoc, XKOS_SOURCE), None)
            tgt = next(g.objects(assoc, XKOS_TARGET), None)
            if src is None or tgt is None:
                continue

            cn_local = _local_name(str(src))       # e.g. "260111000080"
            prodcom_code = _local_name(str(tgt))    # e.g. "07100010"

            cn8 = cn_local[:8]                      # e.g. "26011100"
            hs6 = cn8[:6]                            # e.g. "260111"

            key = _normalize_prodcom(prodcom_code)
            self.prodcom_to_hs6[key].add(hs6)
            self.hs6_to_prodcom[hs6].add(key)
            self.prodcom_to_cn8[key].add(cn8)

    def hs6_for_prodcom(self, prodcom_code: str) -> set:
        """Return the set of HS2022 6-digit subheadings for a Prodcom code."""
        return set(self.prodcom_to_hs6.get(_normalize_prodcom(prodcom_code), set()))

    def prodcom_for_hs6(self, hs6_code: str) -> set:
        """Return the set of Prodcom codes for an HS2022 6-digit subheading."""
        return set(self.hs6_to_prodcom.get(hs6_code.strip(), set()))

In [75]:
PRODCOM_CN_correspondence_table_path = "Prodcom_data/CN2024_PRODCOM2024-export.jsonld"

corr = ProdcomHsCorrespondence(PRODCOM_CN_correspondence_table_path)

print("Prodcom codes loaded:", len(corr.prodcom_to_hs6))
print("HS6 codes loaded:", len(corr.hs6_to_prodcom))

Prodcom codes loaded: 3569
HS6 codes loaded: 5067


### Test with some examples

In [74]:
print("Examples:")

example = "20165130"
print(f"\nHS2022 subheading(s) for Prodcom {example}:", corr.hs6_for_prodcom(example))

# Example with a 1-to-many case (Prodcom code spanning several HS6 headings)
example2 = "10113100"
print(f"HS2022 subheading(s) for Prodcom {example2}:", corr.hs6_for_prodcom(example2))

Examples:

HS2022 subheading(s) for Prodcom 20165130: {'390210'}
HS2022 subheading(s) for Prodcom 10113100: {'020210', '020220', '020230'}


## Import input PRODCOM codes to convert

In [76]:
prodcom_hotspot_product_list = pd.read_csv("tmp/output/20260815 - prodcom codes to HS 2022.csv", dtype={'PRODCOM code':str})
# prodcom_hotspot_product_list

## Convert PRODCOM to HS

In [82]:
max_name_length = 20
for index, prodcom_product in prodcom_hotspot_product_list.iterrows():
    prodcom_code = prodcom_product.get("PRODCOM code")
    # print(prodcom_code)
    HS_codes = corr.hs6_for_prodcom(prodcom_code)
    product_name = prodcom_product.get("Figure name")
    if product_name is None:
        product_name = prodcom2024.loc[prodcom2024["Code"] == prodcom_code].iloc[0]["Explanatory_EN"]
    # short_product_name = "" if type(product_name) is not str else product_name if len(product_name) <= (max_name_length-1) else product_name[:max_name_length-4] + '...'
    print(f" - Prodcom: ({prodcom_code}) \"{product_name}\":", "MULTIPLE CHOICE" if len(HS_codes) > 1 else "NOT FOUND" if len(HS_codes) == 0 else "")
    for code in HS_codes:
        hs_label = cn2024.loc[cn2024["HS_Code"] == code].iloc[0]["Explanatory_EN"]
        print(f"   - HS2022: ({code}) \"{hs_label}\"")
    print("")


 - Prodcom: (20165130) "Polypropylene, in primary forms": 
   - HS2022: (390210) "Polypropylene, in primary forms"

 - Prodcom: (20141130) "Ethylene": 
   - HS2022: (290121) "Ethylene"

 - Prodcom: (20141140) "Propene (propylene)": 
   - HS2022: (290122) "Propene "propylene""

 - Prodcom: (20161050) "Polyethylene having a specific gravity of ≥ 0,94, in primary forms": 
   - HS2022: (390120) "Polyethylene with a specific gravity of >= 0,94, in primary forms"

 - Prodcom: (20142210) "Methanol (methyl alcohol)": 
   - HS2022: (290511) "Methanol "methyl alcohol""

 - Prodcom: (20141223) "Benzene": 
   - HS2022: (290220) "Benzene"

 - Prodcom: (20163010) "Polyvinyl chloride, not mixed with any other substances, in primary forms": 
   - HS2022: (390410) "Poly"vinyl chloride", in primary forms, not mixed with any other substances"

 - Prodcom: (20161039) "Polyethylene having a specific gravity < 0,94, in primary forms (excluding linear)": 
   - HS2022: (390110) "Polyethylene with a specific g